<a href="https://colab.research.google.com/github/daria-bazaliy/Kaggle-competitions/blob/main/restaurant_closure.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import random
from collections import Counter

from datetime import datetime

from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from typing import Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def_train_path = "data/train.csv"
def_test_path = "data/test.csv"

In [ ]:
def read_datasets() -> Tuple[pd.DataFrame, pd.DataFrame]:
    train = pd.read_csv(def_train_path)
    test = pd.read_csv(def_test_path)

    return train, test

In [ ]:
train, submission = read_datasets()
with pd.option_context('display.max_columns', 16):
    print(train.describe(include='all'))


       location_score    rent_index  competition_density    avg_ticket  \
count    16000.000000  16000.000000         16000.000000  16000.000000   
mean         0.677917      0.621874             7.722437    459.548750   
std          0.247641      0.272819             9.994724    238.359098   
min          0.000000      0.000000             0.000000    400.000000   
25%          0.516000      0.433000             1.000000    400.000000   
50%          0.700000      0.643000             3.000000    400.000000   
75%          0.882000      0.846000            11.000000    400.000000   
max          1.000000      1.000000            30.000000   5000.000000   

       chef_experience_years  owner_prev_projects  online_rating_start  \
count           16000.000000         16000.000000         16000.000000   
mean                4.902938             2.440750             2.479544   
std                 3.619214             1.829808             0.656245   
min                 0.000000         

In [ ]:
def_dropping = ['cost_control_score', 'concept_uniqueness', 'avg_ticket', 'weekend_evening_traffic']

In [ ]:
def normalize_dataset(ds: pd.DataFrame) -> pd.DataFrame:
    rand = random.Random(127)
    rd = (lambda x: x * (-rand.random() / 100 + 1))
    def tr(div):
        return lambda x: rd((np.atan(x / div) / np.pi) * 2)

    ds['location_score'] = ds['location_score'].map(lambda x: rd(x))
    ds['rent_index'] = ds['rent_index'].map(lambda x: rd(x))
    ds['competition_density'] = ds['competition_density'].map(tr(7))  # ?
    ds['avg_ticket'] = ds['avg_ticket'].map(tr(500))  # ?
    ds['chef_experience_years'] = ds['chef_experience_years'].map(tr(7))  # ?
    ds['owner_prev_projects'] = ds['owner_prev_projects'].map(tr(7))  # ?
    ds['online_rating_start'] = ds['online_rating_start'].map(lambda x: rd(x / 5.0))
    ds['social_media_score'] = ds['social_media_score'].map(lambda x: rd(x))
    ds['tourist_flow_index'] = ds['tourist_flow_index'].map(lambda x: rd(x))
    ds['weekday_lunch_traffic'] = ds['weekday_lunch_traffic'].map(tr(100))  # ?
    ds['weekend_evening_traffic'] = ds['weekend_evening_traffic'].map(tr(100))  # ?
    ds['staff_turnover_expectation'] = ds['staff_turnover_expectation'].map(lambda x: rd(x))
    ds['concept_uniqueness'] = ds['concept_uniqueness'].map(lambda x: rd(x))
    ds['kitchen_complexity'] = ds['kitchen_complexity'].apply(lambda x: rd(x / 10.0))
    ds['cost_control_score'] = ds['cost_control_score'].map(lambda x: rd(x))
    return ds


In [ ]:
def prepare_dataset(train: pd.DataFrame) -> Tuple[np.ndarray, np.ndarray]:
    train.sample(frac=1, random_state=38)

    classes = train['survived_2y'].map(lambda x: 1 if x == 1 else -1)
    props = train.drop('survived_2y', axis=1)
    props = normalize_dataset(props)
    props = clean_features(props)
    # with pd.option_context('display.max_columns', 16):
    #     print(props.describe(include='all'))

    # print(classes.shape)
    # print(props.shape)

    return props.to_numpy(), classes.to_numpy()


In [ ]:
def clean_features(props: pd.DataFrame) -> pd.DataFrame:
    for df in def_dropping:
        props = props.drop(df, axis=1)
    return props


In [ ]:
class SVM:
    def __init__(self, kernel, degree, gamma, coef0, c):
        self.kernel = kernel
        self.degree = degree
        self.gamma = gamma
        self.coef0 = coef0
        self.c = c
        self.svm = BaggingClassifier(
            estimator=SVC(
                C=c,
                kernel=kernel,
                degree=degree,
                gamma=gamma,
                coef0=coef0,
                probability=True,
                # max_iter=100,
                # verbose = True,
            ),
            oob_score=True,
            n_estimators=64,
            # max_features=11,
            # max_samples=512,
            n_jobs=8,
            random_state=127,
        )

    def name(self):
        return f'svm-kernel-{self.kernel}-degree-{self.degree}-gamma-{self.gamma}-coef0-{self.coef0}-c-{self.c}'

    def fit(self, x_train, y_train):
        self.svm.fit(x_train, y_train)

    def predict(self, x_test):
        idx = np.where(self.svm.classes_ == 1)[0][0]
        preds = self.svm.predict(x_test)
        probs = self.svm.predict_proba(x_test)[:, idx]
        return preds, probs


In [ ]:
import os

x, y = prepare_dataset(train)
x_submission = clean_features(normalize_dataset(submission)).to_numpy()

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.05, random_state=127)


model = SVM('rbf', 3, 1.6, 0.0, 48.0)

model.fit(x_train, y_train)
predictions, probabilities = model.predict(x_test)

acc_nb = np.mean(predictions == y_test)  # вычисляем accuracy (доля правильных предсказаний)
roc_auc_x = roc_auc_score(y_test, probabilities)

print(f'{model.name()}: acc: {acc_nb:.6f}, roc_auc: {roc_auc_x:.6f}')

predictions, probabilities = model.predict(x_submission)
submission = pd.DataFrame(data=probabilities, columns=['survived_2y'])
now = datetime.now().strftime("%Y-%m-%d_%H:%M:%S")

output_dir = 'data/res'
os.makedirs(output_dir, exist_ok=True)

submission.to_csv(
    f'{output_dir}/sub_{roc_auc_x:.5f}_{now}_{model.name()}.csv',
    index_label='id',
    float_format='%.8f',
)


svm-kernel-rbf-degree-3-gamma-1.6-coef0-0.0-c-48.0: acc: 0.958750, roc_auc: 0.977873
